In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam

In [ ]:
from tqdm import tqdm

Design a GRU model (utilize multiple layers and bidirectionality) for sentiment classification on the SST dataset. First create a dataset class for the SST dataset. Train your model on the training set. At the end of each epoch measure the validation accuracy and save the model with the best validation accuracy. Evaluate you model and print the accuracy on the test set

In [ ]:
import pickle
from torch.nn.utils import rnn
from nltk.tokenize import word_tokenize

In [ ]:
with open('train_X.p', 'rb') as fs:
    train_X = pickle.load(fs)

In [ ]:
with open('train_y.p', 'rb') as fs:
    train_y = pickle.load(fs)

In [ ]:
word2index = {}
index2word = {}
i=1
for sent in train_X:
    words = word_tokenize(sent)
    for w in words:
        if w not in word2index:
            word2index[w] = i
            index2word[i] = w
            i+=1

In [ ]:
word2index['<UNK>'] = i
index2word[i] = '<UNK>'

In [ ]:
from torch.utils.data import Dataset, DataLoader

In [ ]:
class SentimentData(Dataset):
    def __init__(self, data, labels):
        self.data = data
        self.labels = labels
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, index):
        sent = self.data[index]
        indices = [word2index[w] if w in word2index else word2index['<UNK>'] for w in word_tokenize(sent)]
        max_length = 50
        length = len(indices) if len(indices)<max_length else max_length
        if len(indices)>max_length:
            indices = indices[:max_length]
        else:
            indices.extend([0 for _ in range(max_length-len(indices))])
        label = self.labels[index]
        return torch.LongTensor(indices), torch.LongTensor([length]), torch.LongTensor([label])

In [ ]:
train_data = SentimentData(train_X, train_y)

In [ ]:
train_loader = DataLoader(train_data, batch_size=24, shuffle=True)

In [ ]:
class Clf_model(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(len(word2index)+1, 300, padding_idx=0)
        self.rnn = nn.GRU(300, 512, batch_first=True, num_layers=2, bidirectional=True)
        self.relu = nn.ReLU()
        self.lin1 = nn.Linear(1024, 128)
        self.lin2 = nn.Linear(128,2)
    
    def forward(self, inp, inp_lengths):
        X = self.emb(inp)
        batch_size = X.shape[0]
        X = rnn.pack_padded_sequence(X, inp_lengths, batch_first=True, enforce_sorted=False)
        h_0 = torch.rand(2*2, batch_size, 512)
        packed_output, last_hidden = self.rnn(X, h_0)
        # last hidden is of shape: 2*num_layers x batch_size x hidden_size
        last_hidden = torch.permute(last_hidden, (1,0,2))
        # After shape changes to: batch_size x 2*num_layers x hidden_size
        # We will utilize only the last hiddenstates (forward + backward)
        last_hidden = last_hidden[:,2:,:].reshape(batch_size, -1) # 2 -> num_layers
        # 0 -> layer 1 forward
        # 1 -> layer 1 backward
        # 2 -> layer 2 forward
        # 3 -> layer 2 backward
        out = self.lin1(last_hidden)
        out = self.relu(out)
        out = self.lin2(out)
        return out

In [ ]:
with open('val_X.p', 'rb') as fs:
    val_X = pickle.load(fs)

In [ ]:
with open('val_y.p', 'rb') as fs:
    val_y = pickle.load(fs)

In [ ]:
val_data = SentimentData(val_X, val_y)

In [ ]:
val_loader = DataLoader(val_data, batch_size=8)

In [ ]:
import wandb
wandb.login(relogin=True)

In [ ]:
wandb.init()

In [ ]:
from sklearn.metrics import accuracy_score

In [ ]:
clf = Clf_model()

In [ ]:
log_interval=25
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(clf.parameters(), lr=0.001)
epochs = 5
for e in tqdm(range(epochs)):
    for batch_idx, (X, l, y) in enumerate(train_loader):
        out = clf(X, l.reshape(-1))
        loss = criterion(out, y.reshape(-1))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if batch_idx % log_interval == 0:
            wandb.log({"loss": loss})
        
    
    clf.eval()
    true = []
    pred = []
    
    with torch.no_grad():
        for X, l, y in val_loader:
            out = clf(X, l.reshape(-1))
            true.extend(y.reshape(-1).numpy().tolist())
            pred.extend(torch.argmax(out, dim=1).numpy().tolist())
        
        acc = accuracy_score(true, pred)
        #print(f"Validation accuracy at epoch {e}: {accuracy_score(true, pred)}")
        wandb.log({"val_accuracy": acc})
    
    clf.train()